In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "legend.fontsize": 13,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "figure.figsize": (7, 5),
})

In [ ]:
# Load data
df_random = pd.read_csv("data/halfmnist_random.csv")
df_irisk  = pd.read_csv("data/halfmnist_instance.csv")

# Extract relevant columns and convert to numeric
for df in [df_random, df_irisk]:
    df["al_labeled_samples"] = pd.to_numeric(df["al_labeled_samples"])
    df["test-c-acc (Max)"]   = pd.to_numeric(df["test-c-acc (Max)"])

# Group by al_labeled_samples and compute mean and std across seeds
df_random_grouped = df_random.groupby("al_labeled_samples")["test-c-acc (Max)"].agg(["mean", "std"]).reset_index()
df_irisk_grouped  = df_irisk.groupby("al_labeled_samples")["test-c-acc (Max)"].agg(["mean", "std"]).reset_index()

# Sort by number of labeled samples
df_random_grouped = df_random_grouped.sort_values("al_labeled_samples").reset_index(drop=True)
df_irisk_grouped  = df_irisk_grouped.sort_values("al_labeled_samples").reset_index(drop=True)

n_seeds_random = df_random.groupby("al_labeled_samples").size().iloc[0]
n_seeds_irisk  = df_irisk.groupby("al_labeled_samples").size().iloc[0]

print(f"Random:        {len(df_random_grouped)} cycles, {n_seeds_random} seeds, samples {df_random_grouped['al_labeled_samples'].min()}-{df_random_grouped['al_labeled_samples'].max()}")
print(f"Instance Risk: {len(df_irisk_grouped)} cycles, {n_seeds_irisk} seeds, samples {df_irisk_grouped['al_labeled_samples'].min()}-{df_irisk_grouped['al_labeled_samples'].max()}")

In [ ]:
# --- Plot: Test Concept Accuracy vs Number of Queries (with std bands) ---
fig, ax = plt.subplots()

x_random = df_random_grouped["al_labeled_samples"].values
y_random_mean = df_random_grouped["mean"].values
y_random_std  = df_random_grouped["std"].values

x_irisk = df_irisk_grouped["al_labeled_samples"].values
y_irisk_mean = df_irisk_grouped["mean"].values
y_irisk_std  = df_irisk_grouped["std"].values

# Plot mean lines
ax.plot(x_random, y_random_mean, marker="o", color="#ff7f0e", label="DPL + random",        linewidth=2, markersize=6)
ax.plot(x_irisk,  y_irisk_mean,  marker="s", color="#9467bd", label="DPL + instance-risk", linewidth=2, markersize=6)

# Plot standard deviation bands
ax.fill_between(x_random, y_random_mean - y_random_std, y_random_mean + y_random_std, 
                color="#ff7f0e", alpha=0.2)
ax.fill_between(x_irisk, y_irisk_mean - y_irisk_std, y_irisk_mean + y_irisk_std, 
                color="#9467bd", alpha=0.2)

ax.set_xlabel("Number of queries")
ax.set_ylabel(r"$Acc_C$")
ax.set_xticks(x_random)
ax.set_ylim(0, 100)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper left")
ax.set_title("HalfMNIST - Active Learning Concept Accuracy", pad=20)

plt.tight_layout()
plt.savefig("../plots/halfmnist_active_learning_concept_acc.pdf", bbox_inches="tight")
plt.show()